# Partial Autocorrelation (PACF) & Reading ACF / PACF Plots

This follows on from the [ACF notebook](9%20Autocorrelation%20%28ACF%29.ipynb). The **ACF** measures the *total* correlation between $y_t$ and $y_{t-k}$. The **PACF** measures the **direct** correlation at lag $k$ after stripping out everything the shorter lags already explain.

## 1. Reading an ACF plot (correlogram)

An **ACF plot** is a stem plot of $\rho_k$ against lag $k$:

- **Lag 0 is always 1** (the tall spike on the left) — a series is perfectly correlated with itself.
- The **shaded band** is the confidence interval, roughly $\pm \dfrac{1.96}{\sqrt{T}}$.
- A spike **inside** the band ≈ statistically **insignificant** (could be noise).
- A spike **outside** the band = **significant** autocorrelation at that lag — "some autocorrelation for lag $k$".

So when the lecture says *"some autocorrelation for lag 1"*, it means the lag-1 stem pokes out past the band.

## 2. What "partial" means

The problem with the plain ACF: if $y_t$ depends on $y_{t-1}$, and $y_{t-1}$ depends on $y_{t-2}$, then $y_t$ looks correlated with $y_{t-2}$ **even if there's no direct link** — the correlation just "leaks" through $y_{t-1}$.

The **Partial Autocorrelation at lag $k$** fixes this: it is the correlation between $y_t$ and $y_{t-k}$ **after removing the linear influence of all the intermediate lags** $y_{t-1}, y_{t-2}, \dots, y_{t-k+1}$.

> *In other words:* ACF = "how related are points $k$ apart, **directly or indirectly**?"  
> PACF = "how related are they once you **discount the steps in between**?"

**As phrased in the lecture:**
- Compute autocorrelation at different lags but **skip the short-term dependencies** (the intermediate lags).
- General implementation idea: **skip the in-between values** when measuring a given lag.
- Use **fewer lag values** — typically only lags up to about **50% of the sample size** are meaningful.

Note $\text{PACF}(0) = 1$ and $\text{PACF}(1) = \text{ACF}(1)$ (at lag 1 there are no intermediate lags to remove).

## 3. A $T=10$ worked example — ACF *and* PACF by hand

Take this complete 10-value series:

| $t$ | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| $y_t$ | 10 | 18 | 12 | 20 | 8 | 16 | 22 | 14 | 11 | 19 |

$$\bar{y} = \frac{150}{10} = 15, \qquad \sum_{t=1}^{10}(y_t-15)^2 = 200$$

### Step 1 — the ACF (same method as notebook 9)

Autocovariance at lag $k$ = $\frac{1}{T}\sum(y_t-\bar y)(y_{t-k}-\bar y)$; divide by the lag-0 value to get $\rho_k$:

| Lag $k$ | $\sum(y_t-15)(y_{t-k}-15)$ | autocov $/T$ | $\rho_k$ (ACF) |
|---|---|---|---|
| 0 | $200$ | $20.0$ | $1.000$ |
| 1 | $-93$ | $-9.3$ | $-0.465$ |
| 2 | $-26$ | $-2.6$ | $-0.130$ |
| 3 | $+17$ | $+1.7$ | $+0.085$ |

### Step 2 — the PACF from the ACF

PACF removes the intermediate lags' influence. For the first lags there are tidy closed forms (the **Durbin–Levinson** recursion):

**Lag 1** — nothing in between, so it equals the ACF:
$$\phi_{11} = \rho_1 = \mathbf{-0.465}$$

**Lag 2** — remove the effect of lag 1:
$$\phi_{22} = \frac{\rho_2 - \rho_1^2}{1 - \rho_1^2} = \frac{-0.130 - (-0.465)^2}{1 - (-0.465)^2} = \frac{-0.130 - 0.2162}{1 - 0.2162} = \frac{-0.3462}{0.7838} = \mathbf{-0.4417}$$

Notice $\phi_{22} \ne \rho_2$: the raw ACF said $-0.130$, but once we discount the strong lag-1 link the **direct** lag-2 effect is a much stronger $-0.442$.

**Lag 3** — remove the effect of lags 1 *and* 2. Durbin–Levinson first updates the lag-1 weight, $\phi_{2,1} = \phi_{11} - \phi_{22}\phi_{11} = -0.6704$, then:
$$\phi_{33} = \frac{\rho_3 - (\phi_{2,1}\rho_2 + \phi_{22}\rho_1)}{1 - (\phi_{2,1}\rho_1 + \phi_{22}\rho_2)} = \mathbf{-0.329}$$

### Result

| Lag $k$ | ACF $\rho_k$ | PACF $\phi_{kk}$ |
|---|---|---|
| 0 | $1.000$ | $1.000$ |
| 1 | $-0.465$ | $-0.465$ &nbsp;(= ACF) |
| 2 | $-0.130$ | $-0.442$ |
| 3 | $+0.085$ | $-0.329$ |

The next cell reproduces every one of these numbers in code.

In [ ]:
import numpy as np
import pandas as pd

y10 = np.array([10, 18, 12, 20, 8, 16, 22, 14, 11, 19], dtype=float)
T = len(y10)
mean = y10.mean()
denom = np.sum((y10 - mean) ** 2)                     # = 200

# --- ACF (biased) ---
def acf_k(k):
    return np.sum((y10[k:] - mean) * (y10[:T - k] - mean)) / denom

rho = [acf_k(k) for k in range(4)]

# --- PACF via the Durbin-Levinson recursion ---
def pacf_durbin_levinson(rho, K):
    phi = {(1, 1): rho[1]}
    out = [1.0, rho[1]]                                # lag 0 and lag 1
    for k in range(2, K + 1):
        num = rho[k] - sum(phi[(k-1, j)] * rho[k-j] for j in range(1, k))
        den = 1     - sum(phi[(k-1, j)] * rho[j]   for j in range(1, k))
        phi[(k, k)] = num / den
        for j in range(1, k):
            phi[(k, j)] = phi[(k-1, j)] - phi[(k, k)] * phi[(k-1, k-j)]
        out.append(phi[(k, k)])
    return out

pacf_vals = pacf_durbin_levinson(rho, 3)

print("mean =", mean, "  sum of squared deviations =", denom)
pd.DataFrame({"lag": range(4),
              "ACF":  np.round(rho, 4),
              "PACF": np.round(pacf_vals, 4)})

In [ ]:
# cross-check against statsmodels (Yule-Walker with biased acovf -> 'ywm')
from statsmodels.tsa.stattools import acf as sm_acf, pacf as sm_pacf

print("statsmodels ACF :", np.round(sm_acf(y10, nlags=3, fft=False), 4))
print("statsmodels PACF:", np.round(sm_pacf(y10, nlags=3, method="ywm"), 4))

## 4. Setup

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf

plt.rcParams["figure.figsize"] = (10, 4)
rng = np.random.default_rng(7)

## 5. Build a series with known structure

We simulate an **AR(2)** process — each value depends *directly* on the previous **two** values:
$$y_t = 0.6\,y_{t-1} - 0.4\,y_{t-2} + \varepsilon_t$$
Because the direct dependence stops at lag 2, the **PACF should cut off after lag 2**, while the **ACF tails off** gradually.

In [ ]:
n = 400
eps = rng.normal(size=n)
y = np.zeros(n)
for t in range(2, n):
    y[t] = 0.6 * y[t-1] - 0.4 * y[t-2] + eps[t]
y = y[50:]          # drop burn-in

plt.plot(y, lw=0.8)
plt.title("Simulated AR(2) series"); plt.xlabel("t"); plt.ylabel("y"); plt.show()

## 6. ACF vs PACF — the numbers

In [ ]:
lags = 10
compare = pd.DataFrame({
    "lag":  range(lags + 1),
    "ACF":  acf(y, nlags=lags, fft=False),
    "PACF": pacf(y, nlags=lags),
}).round(3)
compare

Notice **ACF(1) == PACF(1)** (no intermediate lag to remove), both equal 1 at lag 0, and PACF goes near-zero after lag 2.

## 7. ACF vs PACF — the plots

The stems are the correlations; the shaded band is the confidence interval. Spikes outside the band are significant.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(y,  lags=20, ax=ax1); ax1.set_title("ACF  — tails off gradually")
plot_pacf(y, lags=20, ax=ax2, method="ywm"); ax2.set_title("PACF — cuts off after lag 2")
plt.tight_layout(); plt.show()

## 8. Why we look at both — choosing ARIMA orders

ACF and PACF together are the classic way to pick the **AR order $p$** and **MA order $q$**:

| Pattern | Suggests | Order |
|---|---|---|
| **PACF cuts off** after lag $p$, ACF tails off | **AR($p$)** | $p$ = last significant PACF lag |
| **ACF cuts off** after lag $q$, PACF tails off | **MA($q$)** | $q$ = last significant ACF lag |
| Both tail off | **ARMA($p,q$)** | mix |
| ACF decays *very slowly* | non-stationary → **difference** the series first | — |

Our AR(2) example shows the first row: PACF cuts off after lag 2 → read $p = 2$.

A repeating PACF/ACF spike every $m$ lags (like the **"significant partial autocorrelation for lag 6"** in the slide) points to a **seasonal** dependency of period $m$.

## 9. Summary

- **ACF** = total correlation at lag $k$ (direct + indirect, leaking through intermediate lags).
- **PACF** = the **direct** correlation at lag $k$, with the intermediate lags' influence **removed**.
- Both plots start at 1 (lag 0); spikes outside the confidence band are significant; PACF(1) = ACF(1).
- PACF is built from the ACF via the **Durbin–Levinson** recursion (lag 2: $\phi_{22}=\frac{\rho_2-\rho_1^2}{1-\rho_1^2}$).
- Use only lags up to roughly **half the sample size**.
- The pair drives **ARIMA model selection**: PACF cut-off → AR order $p$; ACF cut-off → MA order $q$; slowly-decaying ACF → difference for stationarity.